# 第8章 車を「動かして」未来を見る (行動条件付き生成 = Forward Dynamics)

第7章は「どうなってほしいか」を文章で伝えて動画を作りました。この章では、文章の代わりに**行動 (action) の数値**を渡します。
「この画像の場所から、車がこう動いたら、カメラにはどう映るか」を予測させるので、**順動力学 (forward dynamics)** と呼びます。
ゲームのように操作を変えると未来が変わる、世界モデルらしい使い方です。

1. 行動の数値の形を知る (公式の軌跡を上から見た図にする)
2. 公式の画像と軌跡で 6 秒先まで予測する
3. 自分で軌跡を作る (直進・左折・右折・停止) → 同じ画像から 4 通りの未来を並べる。自分の画像でも 4 通り作る
4. 自分のドライブレコーダーの画像で試す (操作を 1 つ選ぶ / プロンプトで出来事を足す (実験))
5. 車ではなくカメラを動かす (camera pose)。自分の画像でもカメラを動かす

設定は公式の推奨どおりです: 60 ステップ (10fps で 6 秒) の行動、480p、`num_inference_steps=30`、`guidance_scale=1.0`、`flow_shift=10.0`。
目安: 1 本あたり A100 で 1〜2 分、L4 ではその数倍。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外)。
A100 (40GB 以上) ではモデル全体を GPU に載せ、L4 (24GB) では使う部品だけを GPU に載せる設定 (CPU オフロード) に自動で切り替えます。

**安全フィルタ**: 生成には NVIDIA の安全フィルタ (`cosmos_guardrail`) を使います。使うには Hugging Face で
[nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) の利用申請をして承認を受け、
Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` を登録しておいてください。承認を待たずに試すときは、下の `GUARDRAIL` の印を外します。

In [ ]:
# @title GPU を確かめる
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    GPU_GB = 0
    print(f"CUDA の GPU が見つかりません ({DEVICE} で動かします。とても遅くなります)")

In [ ]:
# @title インストール (数分)
# Cosmos3OmniPipeline は Diffusers の main ブランチに入っている
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title 安全フィルタの設定とサンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

GUARDRAIL = True  # @param {type:"boolean"}
# @markdown 印を付けたまま、安全フィルタの利用申請が通っていないときは、警告を出して安全フィルタなしで続けます

token = None
try:  # Colab のシークレットに HF_TOKEN があれば使う
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN: 読み込みました")
else:
    print("HF_TOKEN: 見つかりません (左の鍵のマークで HF_TOKEN を登録し、「ノートブックからのアクセス」をオンにしてください)")

if GUARDRAIL:
    # 安全フィルタの重みは利用申請が必要なリポジトリにある。生成の途中で止まらないよう、先に読めるか確かめる
    from huggingface_hub import auth_check
    try:
        auth_check("nvidia/Cosmos-1.0-Guardrail", token=token or None)
        print("安全フィルタ: 使えます")
    except Exception as e:
        print(f"安全フィルタ: nvidia/Cosmos-1.0-Guardrail を読めません ({type(e).__name__})")
        print("  - HF_TOKEN が登録されているか")
        print("  - https://huggingface.co/nvidia/Cosmos-1.0-Guardrail で利用申請をして、承認されているか")
        print("  を確かめてください。今回は安全フィルタなしで続けます (GUARDRAIL = False)")
        GUARDRAIL = False

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title 生成モデル (Generator) を読み込む関数
import gc
import json

from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageOps


def load_image(path, max_side=10_000):
    """画像を RGB で読む (max_side より大きければ縮める)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def load_pipe(flow_shift):
    """Cosmos3OmniPipeline を読み込む (2 回目からは読み込み済みのものを使う)。
    flow_shift は公式の推奨値: 画像→動画は 12.0、行動 (action) を使う生成は 10.0"""
    global pipe
    if "pipe" not in globals():
        pipe = Cosmos3OmniPipeline.from_pretrained(
            "nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
        if DEVICE == "cuda" and GPU_GB < 40:
            pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
        else:
            pipe.to(DEVICE)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift, use_karras_sigmas=False)
    return pipe


def free_gpu():
    """読み込んだモデルを GPU から降ろす"""
    for name in ("pipe", "model", "processor"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_video(path, width=640):
    display(Video(path, embed=True, width=width))

In [ ]:
# @title 行動 (action) を作る・描く関数
import numpy as np
import matplotlib.pyplot as plt
from diffusers import CosmosActionCondition

RAW = "https://raw.githubusercontent.com/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
AV_PROMPT = "You are an autonomous vehicle planning system."
# AV の行動は「1 コマ前のカメラから見た、次のコマのカメラの位置と向き」を 9 個の数で表す。
#   [x, y, z, r1..r6]  x: 右, y: 下, z: 前 (OpenCV のカメラ座標)。r1..r6: 回転行列の 1 列目と 2 列目 (rot6d)
# 移動量はメートルを 1.35 倍した値 (公式の可視化コードと同じ換算)
AV_SCALE = 1.35


def rot_y(theta):
    """カメラの下向きの軸 (y) まわりの回転。theta が負だと左を向く"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def from_rot6d(r):
    a = np.asarray(r[:3], float)
    b = np.asarray(r[3:6], float)
    c0 = a / np.linalg.norm(a)
    c1 = b - c0 * (c0 @ b)
    c1 /= np.linalg.norm(c1)
    return np.stack([c0, c1, np.cross(c0, c1)], axis=1)


def make_av_actions(speed_kmh=30.0, left_deg_per_s=0.0, accel=0.0, seconds=6.0, fps=10):
    """速さ (km/h)・左へ曲がる速さ (度/秒、右は負)・加速度 (m/s^2、減速は負) から AV の行動 (T, 9) を作る。
    left_deg_per_s には「時刻 t (秒) → 度/秒」の関数も渡せる (車線変更などに)"""
    rows, dt = [], 1 / fps
    for i in range(int(round(seconds * fps))):
        t = i * dt
        v = max(speed_kmh / 3.6 + accel * t, 0.0)
        d = v * dt
        yaw = left_deg_per_s(t) if callable(left_deg_per_s) else left_deg_per_s
        th = -np.radians(yaw) * dt if d > 0 else 0.0
        move = np.array([np.sin(th / 2) * d, 0.0, np.cos(th / 2) * d]) * AV_SCALE
        R = rot_y(th)
        rows.append(np.concatenate([move, R[:, 0], R[:, 1]]))
    return np.array(rows, np.float32)


def to_path(actions, scale=AV_SCALE):
    """行動 (T, 9) をつなげて、上から見た道筋 (T+1, 3) にする。x: 右, z: 前 (メートル)"""
    R, p, pts = np.eye(3), np.zeros(3), [np.zeros(3)]
    for a in np.asarray(actions, float):
        p = p + R @ (a[:3] / scale)
        R = R @ from_rot6d(a[3:9])
        pts.append(p.copy())
    return np.array(pts)


def plot_paths(paths, title="top-down view", fps=10):
    """{名前: 行動 (T, 9)} を上から見た図に描く。点は 1 秒ごと"""
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, (name, actions) in enumerate(paths.items()):
        p = to_path(actions)
        ax.plot(p[:, 0], p[:, 2], lw=2, label=name)
        ax.scatter(p[::fps, 0], p[::fps, 2], s=12)
    ax.scatter([0], [0], c="k", marker="^", s=80, zorder=5)
    ax.set_xlabel("x: right (m)")
    ax.set_ylabel("z: forward (m)")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(title)
    plt.show()


def tile_videos(videos, labels, out, fps, cols=2):
    """同じ長さの動画 (PIL 画像のリスト) を並べて 1 本にする"""
    from PIL import ImageDraw, ImageFont
    n = min(len(v) for v in videos)
    w, h = videos[0][0].size
    rows = (len(videos) + cols - 1) // cols
    font = ImageFont.load_default(size=max(16, w // 25))
    frames = []
    for t in range(n):
        canvas = Image.new("RGB", (w * cols, h * rows))
        for k, (v, label) in enumerate(zip(videos, labels)):
            tile = v[t].convert("RGB").resize((w, h))
            ImageDraw.Draw(tile).text((10, 8), label, fill="white", font=font, stroke_width=3, stroke_fill="black")
            canvas.paste(tile, ((k % cols) * w, (k // cols) * h))
        frames.append(canvas)
    export_to_video(frames, out, fps=fps, macro_block_size=1)
    return out

## 1. 行動の数値の形を知る

AV (自動運転) の行動は、1 コマごとに 9 個の数です。

| 位置 | 意味 |
| --- | --- |
| 0〜2 | 1 コマ前のカメラから見た移動 (x: 右、y: 下、z: 前)。メートル × 1.35 |
| 3〜8 | 1 コマ前のカメラから見た向きの変化。回転行列の 1 列目と 2 列目 (rot6d) |

60 コマ分つなげると、上から見た車の道筋になります。公式の「直進」「左折」「右折」の 3 本を描いてみます。

In [ ]:
official = {name: np.array(json.loads(Path(fetch(f"{RAW}/generator/action/assets/actions/av_traj_{name}.json")).read_text()), np.float32)
            for name in ("forward", "left", "right")}
print("形:", official["left"].shape, "(コマ数, 9)")
print("左折の最初の 1 コマ:", np.round(official["left"][0], 3))
plot_paths(official, "official AV trajectories (6 s)")

## 2. 公式の画像と軌跡で 6 秒先まで予測する

行動を使うときは、プロンプトは JSON ではなく短い決まり文句 (`You are an autonomous vehicle planning system.`) にします (公式の注意: アップサンプリングしない)。
画像・行動・設定は `CosmosActionCondition` にまとめて渡します。動画の長さは行動の数 + 1 コマ (最初のコマ) で決まります。

In [ ]:
start = load_image(fetch(f"{FW}/generator/action/assets/images/av_0.jpg"))
display(start.resize((416, 240)))


def drive(image, actions, seed=0, prompt=AV_PROMPT):
    """画像と AV の行動 (T, 9) から、T+1 コマの動画 (PIL 画像のリスト) を作る。
    prompt は公式の推奨どおり決まり文句が既定 (4 章の実験でだけ文章を足す)"""
    pipe = load_pipe(flow_shift=10.0)
    result = pipe(
        prompt=prompt,
        action=CosmosActionCondition(
            mode="forward_dynamics",
            chunk_size=len(actions),
            domain_name="av",
            resolution_tier=480,
            raw_actions=torch.as_tensor(actions, dtype=torch.float32),
            image=image,
            view_point="ego_view",
        ),
        fps=10,
        num_inference_steps=30,
        guidance_scale=1.0,
        use_system_prompt=False,
        generator=torch.Generator(device=DEVICE).manual_seed(seed),
    )
    return result.video


frames = drive(start, official["left"])
export_to_video(frames, "ch08_official_left.mp4", fps=10, macro_block_size=1)
show_video("ch08_official_left.mp4")

## 3. 自分で軌跡を作る

`make_av_actions` は、速さ・曲がる速さ・加速度から 60 コマ分の行動を作ります。
同じ最初のコマから 4 通りの操作をして、未来がどう変わるかを並べて比べます。4 本作るので時間がかかります。

曲がる操作は、公式の右折の軌跡に形を寄せてあります。**ゆっくり (15 km/h) 進みながら、最初の 3 秒で 90 度曲がり、そのあとはまっすぐ** という、交差点を曲がる形です。
ゆるいカーブを 6 秒続ける形 (20 km/h・15 度/秒。下の `gentle_turn`) だと、景色に交差点がない画像では、モデルが「道なりに直進」を選びやすくなります。

In [ ]:
def turn_rate(deg_per_s, turn_s=3.0, ramp_s=0.5):
    """交差点を曲がる形: 最初の turn_s 秒だけ deg_per_s 度/秒で曲がり、その後はまっすぐ。
    曲がり始めと終わりの ramp_s 秒は、少しずつ切り替える (急にハンドルを切らない)"""
    def rate(t):
        k = min(1.0, t / ramp_s, max(0.0, (turn_s - t) / ramp_s))
        return deg_per_s * k
    return rate


maneuvers = {
    "straight 40km/h": make_av_actions(speed_kmh=40),
    "turn left": make_av_actions(speed_kmh=15, left_deg_per_s=turn_rate(35)),
    "turn right": make_av_actions(speed_kmh=15, left_deg_per_s=turn_rate(-35)),
    "brake to stop": make_av_actions(speed_kmh=30, accel=-3.0),
}
gentle_turn = {  # 前の版の曲がり方 (ゆるいカーブを 6 秒)。比べたいとき用
    "turn left": make_av_actions(speed_kmh=20, left_deg_per_s=15),
    "turn right": make_av_actions(speed_kmh=20, left_deg_per_s=-15),
}
plot_paths(maneuvers, "my trajectories")

In [ ]:
videos = {}
for name, actions in maneuvers.items():
    videos[name] = drive(start, actions)
    print("done:", name)
tile_videos(list(videos.values()), list(videos), "ch08_four_futures.mp4", fps=10)
show_video("ch08_four_futures.mp4", 832)

「左折」と「右折」で景色が逆に流れ、「停止」では途中から景色が止まれば、行動どおりに未来を描けています。
ただし、交差点でない場所で曲がる指示を出すと、道のない所に進んだり景色が崩れたりします。**世界モデルは見たことのない状況ほど苦手**です。

### 自分の画像でも 4 通りの未来を作る

自分のドライブレコーダーの画像 (車の前方が写った 1 枚) をアップロードして、上と同じ 4 通りの操作で未来を作ります。

画像は 16:9 (832×480) に合わせて中央を切り抜きます。日付や速度の表示が下に入っているときは `crop_bottom` で切り落とすときれいになります。
アップロードした画像は `MY_IMAGE` に入り、この後の 4 章・5 章のセルでも使い回します (別の画像にしたいときは、このセルをもう一度実行します)。

In [ ]:
# @title 自分の画像をアップロードする (colab-only)
from google.colab import files

crop_bottom = 0.0  # @param {type:"slider", min:0, max:0.3, step:0.01}

up = files.upload()
MY_RAW = load_image(next(iter(up)), max_side=10_000)    # 元の画像 (5 章のカメラでも使う)
img = MY_RAW
if crop_bottom > 0:
    img = img.crop((0, 0, img.width, round(img.height * (1 - crop_bottom))))
MY_IMAGE = ImageOps.fit(img, (832, 480), Image.BICUBIC)  # 車の動画用 (16:9 に切り抜き)
display(MY_IMAGE.resize((416, 240)))

**曲がる動画が直進になってしまうとき** は、下のセルの設定を変えてみてください。

- `turn_style`: 曲がる軌跡の種類
  - `strong`: 3 章と同じ「交差点を曲がる形」(既定)
  - `official`: 公式の左折・右折の軌跡そのもの (左折は 30 km/h で 6 秒かけて約 110 度、右折はゆっくり急に約 90 度)
  - `gentle`: 前の版のゆるいカーブ
- `turn_hint`: 印を付けると、曲がる 2 本だけプロンプトに「交差点で左 (右) に曲がる」という英語の文を足します。公式には書かれていない使い方なので実験です
- `seed`: 同じ設定でも、seed を変えると結果が変わります

いちばん効くのは**画像**です。交差点や曲がり角が写っている画像だと曲がりやすく、壁やガードレールに挟まれたまっすぐな道では、どの設定でも直進しがちです。

In [ ]:
# @title 自分の画像で 4 通りの未来を作る (colab-only)
turn_style = "strong"  # @param ["strong", "official", "gentle"]
turn_hint = False  # @param {type:"boolean"}
seed = 0  # @param {type:"integer"}

my_maneuvers = dict(maneuvers)
if turn_style == "official":
    if "official" not in globals():  # 1 章を飛ばしたとき
        official = {n: np.array(json.loads(Path(fetch(f"{RAW}/generator/action/assets/actions/av_traj_{n}.json")).read_text()), np.float32)
                    for n in ("forward", "left", "right")}
    my_maneuvers["turn left"], my_maneuvers["turn right"] = official["left"], official["right"]
elif turn_style == "gentle":
    my_maneuvers.update(gentle_turn)
plot_paths(my_maneuvers, f"turn_style = {turn_style}")

HINTS = {"turn left": "The car turns left at the intersection.", "turn right": "The car turns right at the intersection."}
my_videos = {}
for name, actions in my_maneuvers.items():
    prompt = f"{AV_PROMPT} {HINTS[name]}" if turn_hint and name in HINTS else AV_PROMPT
    my_videos[name] = drive(MY_IMAGE, actions, seed=seed, prompt=prompt)
    print("done:", name)
tile_videos(list(my_videos.values()), list(my_videos), "ch08_my_four_futures.mp4", fps=10)
show_video("ch08_my_four_futures.mp4", 832)

## 4. 自分のドライブレコーダーの画像で試す

操作を 1 つ選んで、自分の画像から 1 本だけ作ります。3 章の最後でアップロードした画像 (`MY_IMAGE`) があればそれを使い、なければここでアップロードします
(切り抜きの `crop_bottom` は、ここでアップロードするときだけ使います)。

In [ ]:
# @title 画像と操作を選ぶ (colab-only)
maneuver = "straight 40km/h"  # @param ["straight 40km/h", "turn left", "turn right", "brake to stop"]
crop_bottom = 0.0  # @param {type:"slider", min:0, max:0.3, step:0.01}

if "MY_IMAGE" not in globals():
    from google.colab import files
    up = files.upload()
    MY_RAW = load_image(next(iter(up)), max_side=10_000)
    img = MY_RAW
    if crop_bottom > 0:
        img = img.crop((0, 0, img.width, round(img.height * (1 - crop_bottom))))
    MY_IMAGE = ImageOps.fit(img, (832, 480), Image.BICUBIC)
display(MY_IMAGE.resize((416, 240)))
frames = drive(MY_IMAGE, maneuvers[maneuver])
export_to_video(frames, "ch08_my_drive.mp4", fps=10, macro_block_size=1)
show_video("ch08_my_drive.mp4")

### プロンプトで出来事を足す (実験)

公式の推奨は、行動を使うときのプロンプトを決まり文句 `You are an autonomous vehicle planning system.` に固定することです。
**文章を足したときのふるまいは公式には書かれていません。** ここでは決まり文句の後ろに、自分で書いた英語の文をつなげて試します。

- `instruction`: 起きてほしい出来事を英語で 1〜2 文 (例: 歩行者が飛び出す、前の車が急に止まる、雨が降り出す)
- `maneuver`: 車の操作 (3 章と同じ 4 通り)
- `use_my_image`: 印を付けると自分の画像 (`MY_IMAGE`) を、外すか画像がなければ 2 章の見本の画像を使う
- `compare`: 印を付けると、決まり文句だけの動画も作って横に並べる。文章が効いているかを見比べられる (2 本作るので時間は 2 倍)

もっと自由に出来事を文章で指示したいときは、操作を使わずに文章だけで作る単独のノートブック `dashcam_whatif.ipynb` も使えます。

In [ ]:
# @title プロンプトで指示して作る (実験, colab-only)
instruction = "A pedestrian suddenly steps out and crosses the road in front of the car."  # @param {type:"string"}
maneuver = "straight 40km/h"  # @param ["straight 40km/h", "turn left", "turn right", "brake to stop"]
use_my_image = True  # @param {type:"boolean"}
compare = True  # @param {type:"boolean"}
seed = 0  # @param {type:"integer"}

image = MY_IMAGE if use_my_image and "MY_IMAGE" in globals() else start
if use_my_image and image is start:
    print("自分の画像 (MY_IMAGE) がないので、見本の画像を使います")
if not instruction.isascii():
    print("注意: instruction は英語で書くと効きやすくなります")
prompt = f"{AV_PROMPT} {instruction.strip()}"
print("プロンプト:", prompt)

videos = [drive(image, maneuvers[maneuver], seed=seed, prompt=prompt)]
labels = ["with your prompt"]
if compare:
    videos.append(drive(image, maneuvers[maneuver], seed=seed))
    labels.append("fixed prompt only")
tile_videos(videos, labels, "ch08_my_prompt.mp4", fps=10)
show_video("ch08_my_prompt.mp4", 832)

## 5. 車ではなくカメラを動かす

`domain_name="camera_pose"` にすると、同じ 9 個の数の形で「カメラの動き」を渡せます。風景写真の中をカメラが飛んでいくような動画になります。
こちらは 30fps で、プロンプトは場面を説明する短い文章です。公式の灯台の例と、その動きを左右反転した例を並べます。

In [ ]:
cam = np.array(json.loads(Path(fetch(f"{RAW}/generator/action/assets/actions/camera_action.json")).read_text()), np.float32)
S = np.diag([-1.0, 1.0, 1.0])  # 左右反転: 移動の x の符号を変え、回転を S R S にする
mirrored = cam.copy()
for i, a in enumerate(cam):
    R = S @ from_rot6d(a[3:9]) @ S
    mirrored[i] = np.concatenate([S @ a[:3], R[:, 0], R[:, 1]])

lighthouse = load_image(fetch(f"{FW}/generator/action/assets/images/lighthouse_720.png"), max_side=10_000)
prompt = Path(fetch(f"{RAW}/generator/action/assets/prompts/lighthouse.txt")).read_text().strip()
print(prompt)

pipe = load_pipe(flow_shift=10.0)
cams = []
for actions in (cam, mirrored):
    result = pipe(
        prompt=prompt,
        action=CosmosActionCondition(mode="forward_dynamics", chunk_size=len(actions), domain_name="camera_pose",
                                     resolution_tier=480, raw_actions=torch.as_tensor(actions), image=lighthouse,
                                     view_point="ego_view"),
        fps=30, num_inference_steps=30, guidance_scale=1.0, use_system_prompt=False,
        generator=torch.Generator(device=DEVICE).manual_seed(0),
    )
    cams.append(result.video)
tile_videos(cams, ["official", "mirrored"], "ch08_camera.mp4", fps=30)
show_video("ch08_camera.mp4", 832)

### 自分の画像でカメラを動かす

自分の写真 (風景・街並み・部屋など) の中で、カメラを動かした動画を作ります。動きは最大 4 つまで選べて、並べて比べられます。
画像は 3 章の最後でアップロードした元の画像 (`MY_RAW`) を使います。別の画像にしたいとき、またはまだアップロードしていないときは `new_image` に印を付けます。
画像は公式の例と同じ 16:9 (1280×720) に中央を切り抜きます。

- `scene`: 写っている場面を英語で短く説明する文。選んだ動きを説明する英語の文 (例: `The camera slowly moves forward.`) が後ろに自動で付きます
- `move1`〜`move4`: カメラの動き。「(なし)」にすると作りません
- `speed`: 動きの大きさ。1.0 が公式の例と同じくらい (1 コマあたりの移動 約 0.9)

| 動き | 中身 |
| --- | --- |
| official / mirrored | 公式の `camera_action.json` とその左右反転。公式のノートブックも、この同じ動きを灯台・テラス・山の 3 つの場面に使っています |
| dolly in / out | 前へ進む / 後ろへ下がる |
| truck left / right | 向きを変えずに左 / 右へ平行に動く |
| crane up / down | 上へ昇る / 下へ降りる |
| pan left / right | その場で左 / 右を向く (1 コマ 0.5 度、2 秒で 30 度) |
| orbit left / right | 左 / 右へ動きながら、真ん中を見続けるように向きを変える (公式の動きと同じ形) |

公式の README によると、カメラの動きは AV と同じ表し方 (x: 右、y: 下、z: 前の移動と rot6d の回転) です。
**official と mirrored 以外は、この軸の向きと、公式の例の大きさをもとに、ここで作った動きです (実験)。** 思った向きに動かないときは、下の「うまくいかないとき」を見てください。
1 つの動き (30fps で 60 コマ = 2 秒) につき、A100 で 1〜2 分かかります。

In [ ]:
# @title 自分の画像でカメラを動かす (colab-only)
scene = "A realistic photo of a quiet street lined with houses and trees on a sunny day."  # @param {type:"string"}
move1 = "official"  # @param ["(なし)", "official", "mirrored", "dolly in", "dolly out", "truck left", "truck right", "crane up", "crane down", "pan left", "pan right", "orbit left", "orbit right"]
move2 = "dolly in"  # @param ["(なし)", "official", "mirrored", "dolly in", "dolly out", "truck left", "truck right", "crane up", "crane down", "pan left", "pan right", "orbit left", "orbit right"]
move3 = "pan right"  # @param ["(なし)", "official", "mirrored", "dolly in", "dolly out", "truck left", "truck right", "crane up", "crane down", "pan left", "pan right", "orbit left", "orbit right"]
move4 = "crane up"  # @param ["(なし)", "official", "mirrored", "dolly in", "dolly out", "truck left", "truck right", "crane up", "crane down", "pan left", "pan right", "orbit left", "orbit right"]
speed = 1.0  # @param {type:"slider", min:0.25, max:2, step:0.25}
new_image = False  # @param {type:"boolean"}
seed = 0  # @param {type:"integer"}

if new_image or "MY_RAW" not in globals():
    from google.colab import files
    up = files.upload()
    MY_RAW = load_image(next(iter(up)), max_side=10_000)
cam_image = ImageOps.fit(MY_RAW, (1280, 720), Image.BICUBIC)
display(cam_image.resize((416, 234)))

# 公式の動き (前のセルを飛ばしても動くよう、ここでも読み込む)。1 コマあたり約 (-0.9, 0, +0.1) の移動と、右へ約 0.29 度の回転
cam = np.array(json.loads(Path(fetch(f"{RAW}/generator/action/assets/actions/camera_action.json")).read_text()), np.float32)
S = np.diag([-1.0, 1.0, 1.0])  # 左右反転: 移動の x の符号を変え、回転を S R S にする
mirrored = np.array([np.concatenate([S @ a[:3], (S @ from_rot6d(a[3:9]) @ S)[:, 0], (S @ from_rot6d(a[3:9]) @ S)[:, 1]])
                     for a in cam], np.float32)

# 作った動き: (1 コマの移動の向き (x: 右, y: 下, z: 前), 1 コマに右へ向く角度 (度), プロンプトに足す文)
CAMERA_MOVES = {
    "dolly in":    ((0, 0, 1),  0.0,  "The camera slowly moves forward."),
    "dolly out":   ((0, 0, -1), 0.0,  "The camera slowly pulls back."),
    "truck left":  ((-1, 0, 0), 0.0,  "The camera slowly slides to the left."),
    "truck right": ((1, 0, 0),  0.0,  "The camera slowly slides to the right."),
    "crane up":    ((0, -1, 0), 0.0,  "The camera slowly rises upward."),      # y は下が正なので、上は負
    "crane down":  ((0, 1, 0),  0.0,  "The camera slowly lowers downward."),
    "pan left":    ((0, 0, 0),  -0.5, "The camera slowly pans to the left."),  # rot_y は正が右向き
    "pan right":   ((0, 0, 0),  0.5,  "The camera slowly pans to the right."),
    "orbit left":  ((-1, 0, 0), 0.29, "The camera slowly orbits to the left around the scene."),
    "orbit right": ((1, 0, 0),  -0.29, "The camera slowly orbits to the right around the scene."),
}
STEADY = "The camera moves slowly and smoothly through the scene."


def camera_actions(move, speed=1.0, frames=60):
    """動きの名前 → (カメラの行動 (frames, 9), プロンプトに足す文)"""
    if move in ("official", "mirrored"):
        a = (cam if move == "official" else mirrored).copy()
        a[:, :3] *= speed                        # 公式の動きは、移動の大きさだけ speed 倍にする
        return a, STEADY
    direction, yaw_deg, sentence = CAMERA_MOVES[move]
    V = 0.9 * speed                              # 1 コマの移動量 (公式の例と同じくらい)
    R = rot_y(np.radians(yaw_deg * speed))
    row = np.concatenate([np.array(direction, float) * V, R[:, 0], R[:, 1]])
    return np.tile(row, (frames, 1)).astype(np.float32), sentence


pipe = load_pipe(flow_shift=10.0)
moves = [m for m in (move1, move2, move3, move4) if m != "(なし)"]
my_cams = []
for move in moves:
    actions, sentence = camera_actions(move, speed)
    prompt = f"{scene.strip()} {sentence}"
    print(f"{move}: {prompt}")
    result = pipe(
        prompt=prompt,
        action=CosmosActionCondition(mode="forward_dynamics", chunk_size=len(actions), domain_name="camera_pose",
                                     resolution_tier=480, raw_actions=torch.as_tensor(actions), image=cam_image,
                                     view_point="ego_view"),
        fps=30, num_inference_steps=30, guidance_scale=1.0, use_system_prompt=False,
        generator=torch.Generator(device=DEVICE).manual_seed(seed),
    )
    my_cams.append(result.video)
if len(my_cams) > 1:
    tile_videos(my_cams, moves, "ch08_my_camera.mp4", fps=30)
elif my_cams:
    export_to_video(my_cams[0], "ch08_my_camera.mp4", fps=30, macro_block_size=1)
if my_cams:
    show_video("ch08_my_camera.mp4", 832)
else:
    print("move1〜move4 がすべて「(なし)」です")

## うまくいかないとき

- **行動どおりに動かない**: 速すぎる・急すぎる操作は学習データにないので無視されがちです。まずは 20〜50 km/h から試してください。曲がるときは逆に、ゆっくり進みながら大きく曲がる (15 km/h・35 度/秒を 3 秒) ほうが、交差点を曲がる映像として描かれやすくなります
- **曲がるはずが直進になる**: 交差点や曲がり角が写った画像を使う、3 章の自分の画像のセルで `turn_style` を `official` にする、`turn_hint` に印を付ける、seed を変える、の順に試してください
- **景色が崩れる**: seed を変えて作り直すか、より道路が広く写った画像を使います
- **時間がかかる**: 行動を 30 コマ (`seconds=3`) に減らすと速くなります
- **プロンプトの出来事が起きない**: 決まり文句だけの動画と見比べて、差がなければ文章は効いていません。出来事を 1 つに絞って具体的に書くか、seed を変えます。それでも起きないときは `dashcam_whatif.ipynb` (文章だけで作る) を使います
- **カメラが思った向きに動かない**: official / mirrored 以外の動きは、軸の向きを仮定して作った実験です。左右や上下が逆に見えたら、反対の動き (例: `pan left` と `pan right`) を選んでください。ほとんど動かないときは `speed` を上げ、景色が崩れるときは下げます
- **カメラの動きと文が合わない**: `scene` には場面だけを書き、カメラの動きの言葉 (zoom、pan など) は書かないでください。動きの文は自動で足されるので、食い違うとどちらにも動かなくなりがちです
- 作られる動画はもっともらしい予測で、本当にその道を走ったときの映像ではありません。第9章では、この予測動画から逆に軌跡を当て直して、どれくらい行動どおりかを確かめます